# Unit Test Generation — Autoresearch
PhD Research: Plain LLM vs Simple RAG vs Iterative Critique RAG for unit test generation.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Install & start Ollama
4. Clone repo
5. One-time setup (dataset + knowledge base)
6. Run a single experiment (quick test)
7. GitHub credentials (run once before Step 8)
8. Run all 12 experiments × 3 models (full PhD comparison)
9. Generalizability analysis (do rankings hold across models?)
10. Visualize results
11. Cross-task comparison (RQ4)
12. Final push to GitHub

## Step 1: Mount Google Drive
**Run this first.** All results, logs, charts, and checkpoints are saved to both local Colab storage and Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# All PhD outputs go here — edit if you want a different Drive folder
DRIVE_DIR = '/content/drive/MyDrive/PhD_autoresearch'

LOGS_DIR        = os.path.join(DRIVE_DIR, 'logs')
PLOTS_DIR       = os.path.join(DRIVE_DIR, 'plots')
RESULTS_DIR     = os.path.join(DRIVE_DIR, 'results')
CHECKPOINTS_DIR = os.path.join(DRIVE_DIR, 'checkpoints')

for d in [LOGS_DIR, PLOTS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

def save_to_both(src, filename, subdir='results'):
    """Copy a file to both local repo dir and Drive."""
    if not os.path.exists(src):
        print(f'  SKIP {filename} — not found')
        return
    drive_path = os.path.join(DRIVE_DIR, subdir, filename)
    shutil.copy(src, drive_path)
    print(f'  {filename} → Drive')

print('Drive mounted.')
print(f'  Drive: {DRIVE_DIR}/')

## Step 2: Install Python dependencies

In [ ]:
!pip install -q ollama datasets sentence-transformers scikit-learn nltk rouge beautifulsoup4 numpy pandas matplotlib scipy

## Step 3: Install Ollama and pull models
Pulls all 3 models needed for the multi-model experiment.
On A100 (40GB): all 3 fit comfortably.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started.')

In [ ]:
# Models for multi-model generalizability study
# A100 (40GB): all 3 fit
# T4  (15GB):  use llama3.2:latest only

MODELS = [
    'llama3.2:latest',   # 3B  — fast baseline
    'phi4:14b',          # 14B — mid-size
    'qwen2.5:14b',       # 14B — mid-size
]

for model in MODELS:
    print(f'\nPulling {model}...')
    !ollama pull {model}

print('\nAll models ready.')
!ollama list

## Step 4: Clone the repo
Uses `fetch + reset --hard` instead of `git pull` to avoid merge conflicts
when `train_unitest.py` has been patched locally by a previous session.

In [ ]:
REPO_DIR = '/content/autoresearch'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/balajivenky06/autoresearch.git {REPO_DIR}
else:
    # reset --hard prevents git pull conflicts when train_unitest.py was
    # locally patched by set_experiment() in a previous session.
    # Untracked files (results_unitest.tsv, plots/, .checkpoints/) are NOT affected.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/master

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 5: One-time setup (download dataset + build knowledge base)
Only needs to run once per Colab session.

In [ ]:
!python prepare_unitest.py

## Step 6: Configure and run a single experiment (quick test)
Use this to verify everything works before running the full multi-model sweep.

In [ ]:
import re, ast

VALID_METHODS    = {'plain_llm', 'simple_rag', 'iterative_critique'}
VALID_REASONINGS = {'base', 'cot', 'tot', 'got'}

def set_experiment(method='plain_llm', reasoning='base', model=MODELS[0]):
    """Update train_unitest.py config in-place."""
    assert method in VALID_METHODS, f'method must be one of {VALID_METHODS}'
    assert reasoning in VALID_REASONINGS, f'reasoning must be one of {VALID_REASONINGS}'

    with open('train_unitest.py', 'r') as f:
        content = f.read()

    content = re.sub(r'^METHOD\s*=.*$',         'METHOD    = "' + method    + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^REASONING\s*=.*$',       'REASONING = "' + reasoning + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^GENERATOR_MODEL\s*=.*$', 'GENERATOR_MODEL = "' + model + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^HELPER_MODEL\s*=.*$',    'HELPER_MODEL    = "' + model + '"', content, flags=re.MULTILINE)
    # Always enforce full dataset — guards against accidental MAX_SAMPLES override
    content = re.sub(r'^MAX_SAMPLES\s*=.*$',     'MAX_SAMPLES          = None', content, flags=re.MULTILINE)

    try:
        ast.parse(content)
    except SyntaxError as e:
        raise RuntimeError('set_experiment produced invalid Python: ' + str(e))

    with open('train_unitest.py', 'w') as f:
        f.write(content)
    print(f'Config set: method={method}, reasoning={reasoning}, model={model}')

# Quick test with first model
set_experiment(method='plain_llm', reasoning='base', model=MODELS[0])

In [ ]:
log_local = 'run_unitest.log'
!python train_unitest.py 2>&1 | tee {log_local}

shutil.copy(log_local, os.path.join(LOGS_DIR, 'single_run.log'))
save_to_both('results_unitest.tsv', 'results_unitest.tsv')

!grep -E '^val_score:|^method:|^model:|^avg_|^Results appended' {log_local}

## Step 7: GitHub credentials
**Run this once before Step 8.**  
Sets up git identity and unblocks `results_unitest.tsv` from `.gitignore`  
so incremental pushes work automatically during the experiment loop.

Get a token at: GitHub → Settings → Developer Settings → Personal Access Tokens → New token (scope: `repo`)

In [ ]:
# ---- Set your GitHub credentials ----
GITHUB_TOKEN = 'ghp_xxxxxxxxxxxxxxxxxxxx'   # paste your token here
GITHUB_EMAIL = 'your@email.com'
GITHUB_NAME  = 'Your Name'
# -------------------------------------

import subprocess

subprocess.run(['git', 'config', 'user.email', GITHUB_EMAIL], capture_output=True)
subprocess.run(['git', 'config', 'user.name',  GITHUB_NAME],  capture_output=True)

# Unblock results_unitest.tsv and logs from .gitignore once — persists for all auto-pushes
with open('.gitignore', 'r') as f:
    gi = f.read()
changed = False
if 'results_unitest.tsv' in gi and '# results_unitest.tsv' not in gi:
    gi = gi.replace('results_unitest.tsv', '# results_unitest.tsv  (committed for PhD)')
    changed = True
if 'summary_all_experiments.csv' in gi and '# summary_all_experiments' not in gi:
    gi = gi.replace('summary_all_experiments.csv', '# summary_all_experiments.csv  (committed for PhD)')
    changed = True
if changed:
    with open('.gitignore', 'w') as f:
        f.write(gi)
    print('Unblocked results files from .gitignore')

print('GitHub credentials set. Ready for auto-push in Step 8.')

## Step 8: Run all 12 experiments × 3 models
**Full PhD comparison: 36 runs total (~4–7 hours on A100)**

- 3 models × 3 methods × 4 reasoning = 36 experiments
- Results, logs, and checkpoints saved to **Drive after every experiment**
- Results and logs **pushed to GitHub every 5 experiments**

**On runtime reset:** re-run Steps 1–5 and Step 7, then re-run this cell.
- Already-completed experiments are **automatically skipped** (read from Drive TSV)
- Partially-completed experiments resume from their last sample (Drive checkpoint)

In [ ]:
import subprocess, pandas as pd

EXPERIMENTS = [
    ('plain_llm',          'base'),
    ('plain_llm',          'cot'),
    ('plain_llm',          'tot'),
    ('plain_llm',          'got'),
    ('simple_rag',         'base'),
    ('simple_rag',         'cot'),
    ('simple_rag',         'tot'),
    ('simple_rag',         'got'),
    ('iterative_critique', 'base'),
    ('iterative_critique', 'cot'),
    ('iterative_critique', 'tot'),
    ('iterative_critique', 'got'),
]

# -----------------------------------------------------------------------
# Incremental git push — called every 5 experiments and at the final run
# -----------------------------------------------------------------------
def auto_push(run_num, total_runs):
    """Stage results + logs and push to GitHub. Silently skips if nothing new."""
    # Stage results TSV
    subprocess.run(['git', 'add', '-f', 'results_unitest.tsv'], capture_output=True)
    # Stage all log files produced so far
    subprocess.run('git add -f *.log 2>/dev/null || true', shell=True, capture_output=True)
    # Stage checkpoints directory if it exists
    subprocess.run('git add -f .checkpoints/ 2>/dev/null || true', shell=True, capture_output=True)

    msg = f'progress [{run_num}/{total_runs}]: results + logs after {run_num} experiments'
    commit = subprocess.run(
        ['git', 'commit', '-m', msg],
        capture_output=True, text=True
    )
    if 'nothing to commit' in (commit.stdout + commit.stderr):
        print(f'  git: nothing new to commit [{run_num}/{total_runs}]')
        return

    remote_url = f'https://{GITHUB_TOKEN}@github.com/balajivenky06/autoresearch.git'
    push = subprocess.run(
        ['git', 'push', remote_url, 'master'],
        capture_output=True, text=True
    )
    if push.returncode == 0:
        print(f'  git push OK [{run_num}/{total_runs}]')
    else:
        print(f'  git push FAILED: {push.stderr[:120]}')

# -----------------------------------------------------------------------
# Restore results_unitest.tsv from Drive if local copy is missing.
# This recovers all completed experiment rows after a VM reset.
# -----------------------------------------------------------------------
local_tsv = 'results_unitest.tsv'
drive_tsv = os.path.join(RESULTS_DIR, 'results_unitest.tsv')

if not os.path.exists(local_tsv) and os.path.exists(drive_tsv):
    shutil.copy(drive_tsv, local_tsv)
    print(f'Restored results_unitest.tsv from Drive.')

# Build a set of already-completed (method, model) pairs so we can skip them.
completed  = set()   # {('plain_llm/base', 'llama3.2:latest'), ...}
prior_rows = {}      # key → row dict, for loading into all_results

if os.path.exists(local_tsv):
    try:
        prior_df = pd.read_csv(local_tsv, sep='\t')
        for _, r in prior_df.iterrows():
            if str(r.get('status', '')) == 'ok':
                key = (str(r['method']), str(r['model']))
                completed.add(key)
                prior_rows[key] = r.to_dict()
        print(f'Found {len(completed)} already-completed experiments — will skip them.')
    except Exception as e:
        print(f'Could not read existing TSV ({e}) — starting fresh.')

# -----------------------------------------------------------------------
# Main experiment loop
# -----------------------------------------------------------------------
all_results = []
sep = '=' * 60
total_runs = len(MODELS) * len(EXPERIMENTS)
run_num = 0

for model in MODELS:
    print(f'\n{sep}')
    print(f'  MODEL: {model}')
    print(sep)

    for method, reasoning in EXPERIMENTS:
        run_num += 1
        method_key = f'{method}/{reasoning}'
        print(f'\n  [{run_num}/{total_runs}] {method_key} — {model}')

        # Skip if this experiment already completed in a previous session
        if (method_key, model) in completed:
            print(f'    SKIP — already completed (found in results_unitest.tsv)')
            r = prior_rows[(method_key, model)]
            all_results.append({
                'model': model, 'method': method_key, 'status': 'ok',
                'val_score': float(r.get('val_score', 0)),
            })
            continue

        set_experiment(method=method, reasoning=reasoning, model=model)

        log_local = f'{model.replace(":", "-")}_{method}_{reasoning}.log'
        try:
            ret = subprocess.run(
                ['python', 'train_unitest.py'],
                capture_output=True, text=True, timeout=900
            )
            output = ret.stdout + ret.stderr
            with open(log_local, 'w') as f:
                f.write(output)
            # Save log to Drive
            shutil.copy(log_local, os.path.join(LOGS_DIR, log_local))

        except subprocess.TimeoutExpired:
            print('    TIMEOUT — skipping')
            all_results.append({
                'model': model, 'method': method_key,
                'val_score': 0.0, 'status': 'timeout'
            })
            continue

        # Parse key metrics from stdout for in-memory summary
        row = {'model': model, 'method': method_key, 'status': 'ok'}
        for line in output.splitlines():
            for key in ['val_score', 'avg_noise_rate', 'avg_faithfulness',
                        'avg_retrieval_secs', 'avg_llm_secs', 'avg_syntax',
                        'avg_edge', 'avg_assert_density', 'avg_semantic_sim',
                        'avg_rouge']:
                if line.strip().startswith(key + ':'):
                    try:
                        row[key] = float(line.split(':')[1].strip())
                    except Exception:
                        pass
        if row.get('val_score', 0.0) == 0.0:
            row['status'] = 'crash'
        all_results.append(row)
        print(f'    val_score: {row.get("val_score", 0):.4f}  [{row["status"]}]')

        # Save results TSV + active checkpoint to Drive after every experiment
        save_to_both('results_unitest.tsv', 'results_unitest.tsv')
        if os.path.isdir('.checkpoints'):
            for ckpt in os.listdir('.checkpoints'):
                shutil.copy(
                    os.path.join('.checkpoints', ckpt),
                    os.path.join(CHECKPOINTS_DIR, ckpt)
                )

        # Push results + logs to GitHub every 5 experiments and on the final run
        if run_num % 5 == 0 or run_num == total_runs:
            print(f'  --- auto-push [{run_num}/{total_runs}] ---')
            auto_push(run_num, total_runs)

# -----------------------------------------------------------------------
# Final summary
# -----------------------------------------------------------------------
df = pd.DataFrame(all_results)
print(f'\n{sep}')
print('  FINAL RESULTS — ALL MODELS')
print(sep)
if not df.empty and 'val_score' in df.columns:
    pivot = df.pivot_table(
        index='method', columns='model', values='val_score', aggfunc='max'
    ).round(4)
    print(pivot.to_string())

df.to_csv('summary_all_experiments.csv', index=False)
save_to_both('summary_all_experiments.csv', 'summary_all_experiments.csv')
print('\nDone.')

## Step 9: Generalizability analysis
**Key PhD question: do method rankings hold consistently across all 3 models?**

Computes Spearman rank correlation between models.
- ρ ≥ 0.8 across all pairs → findings generalize (model-agnostic)
- ρ < 0.8 → rankings are model-dependent (important nuance for thesis)

Outputs 4 charts + a written report for the thesis appendix.

In [ ]:
from IPython.display import Image, display

!python analyze_generalizability.py

generalizability_charts = [
    'val_score_by_model.png',
    'faithfulness_by_model.png',
    'rank_stability.png',
    'rank_correlation.png',
]

for fname in generalizability_charts:
    src = os.path.join('plots_generalizability', fname)
    save_to_both(src, f'generalizability_{fname}', subdir='plots')
    if os.path.exists(src):
        display(Image(src))

save_to_both(
    'plots_generalizability/generalizability_report.txt',
    'generalizability_report.txt'
)

## Step 10: Visualize results
Generates 10 KPI charts (7 single-model + 3 cross-model) — saved to both local and Drive.

In [ ]:
from IPython.display import Image, display

!python visualize_unitest.py

charts = [
    'heatmap.png', 'grouped_bar.png', 'radar.png', 'per_metric_bar.png',
    'noise_rate.png', 'cost_breakdown.png', 'faithfulness.png',
    'model_val_score.png', 'model_faithfulness.png', 'model_rank_stability.png',
]

for fname in charts:
    src = os.path.join('plots_unitest', fname)
    save_to_both(src, fname, subdir='plots')
    if os.path.exists(src):
        display(Image(src))

## Step 11: Cross-task comparison — RQ4
Requires `results_docstring.tsv` generated locally by `extract_docstring_results.py`.

**Before running this step:**
1. On your local machine: `python extract_docstring_results.py`
2. Upload the generated `results_docstring.tsv` to `PhD_autoresearch/results/` in Drive
3. Then run this cell.

In [ ]:
# Load results_docstring.tsv from Drive
docstring_tsv_drive = os.path.join(RESULTS_DIR, 'results_docstring.tsv')
if os.path.exists(docstring_tsv_drive):
    shutil.copy(docstring_tsv_drive, 'results_docstring.tsv')
    print('results_docstring.tsv loaded from Drive.')
else:
    print(f'Not found: {docstring_tsv_drive}')
    print('Run extract_docstring_results.py locally, upload to Drive, then re-run this cell.')

In [ ]:
from IPython.display import Image, display

!python compare_tasks.py

compare_charts = ['faithfulness_by_task.png', 'noise_vs_faithfulness.png', 'pareto.png']
for fname in compare_charts:
    src = os.path.join('plots_compare', fname)
    save_to_both(src, f'compare_{fname}', subdir='plots')
    if os.path.exists(src):
        display(Image(src))

save_to_both('plots_compare/summary_table.tsv', 'cross_task_summary.tsv')

## Step 12: Final push to GitHub
Commits all plots, charts, and the final summary — everything not already in git.

In [ ]:
import subprocess

# Stage all remaining outputs not yet pushed
for f in ['results_unitest.tsv', 'summary_all_experiments.csv',
          'plots_unitest/', 'plots_compare/', 'plots_generalizability/']:
    subprocess.run(f'git add -f {f} 2>/dev/null || true', shell=True, capture_output=True)

models_str = ', '.join(MODELS)
commit_msg = f'final results: {models_str} — 36 runs (3 models × 12 methods) + all charts'
result = subprocess.run(['git', 'commit', '-m', commit_msg], capture_output=True, text=True)
print(result.stdout or result.stderr)

remote_url = f'https://{GITHUB_TOKEN}@github.com/balajivenky06/autoresearch.git'
push = subprocess.run(['git', 'push', remote_url, 'master'], capture_output=True, text=True)
if push.returncode == 0:
    print('All outputs pushed to GitHub successfully.')
else:
    print('Push failed:', push.stderr)